# SageMaker Setup and Verification

**Time**: 15 minutes  
**Platform**: Amazon SageMaker (Studio Lab, Notebook Instance, or Studio)

This notebook will help you:
- Install required packages in SageMaker
- Configure credentials (using IAM roles + Secrets Manager)
- Verify Elastic Cloud connection
- Verify AWS Bedrock access (using IAM role)
- Test ELSER model

---

## ✨ SageMaker Benefits

✅ **No local setup** - Everything in AWS  
✅ **IAM integration** - Automatic AWS credentials  
✅ **Low latency** - Same region as Bedrock & Elastic  
✅ **Cost effective** - ~$0.05/hour for ml.t3.medium  
✅ **Persistent** - Work saved automatically  

---

## Step 1: Install Required Packages

**Note**: This installs packages in your current SageMaker environment.

In [ ]:
# Install workshop dependencies
import sys
import subprocess

print("📦 Installing workshop dependencies...\n")

packages = [
    'elasticsearch==8.15.0',
    'boto3==1.34.70',
    'elastic-apm==6.21.0',
    'python-dotenv==1.0.1',
    'requests==2.31.0'
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', package
    ])

print("\n✅ All packages installed successfully!")
print("\n🎯 Ready to configure credentials →")

## Step 2: Configure Credentials

### 🔒 SageMaker offers multiple credential options:

1. **IAM Role** (automatic for AWS) - ✅ Recommended
2. **Secrets Manager** (secure for Elastic) - ✅ Recommended for production
3. **Direct configuration** (easiest for workshop)

### Choose your approach below:

In [ ]:
import os
import boto3
import json

print("🔧 CREDENTIAL CONFIGURATION\n")
print("="*60)

# ===== AWS CREDENTIALS =====
# SageMaker provides these automatically via IAM role!
os.environ['AWS_REGION'] = 'us-east-1'  # Change if your Elastic is in different region

print("✅ AWS Credentials: Using IAM role (automatic in SageMaker)")
print("   Region: us-east-1")

# ===== ELASTIC CREDENTIALS =====
print("\n📋 Elastic Cloud Configuration:")
print("-" * 60)

# Option 1: Try to load from AWS Secrets Manager (most secure)
USE_SECRETS_MANAGER = True  # Set to False to use direct configuration
SECRET_NAME = 'elastic-workshop-credentials'

if USE_SECRETS_MANAGER:
    try:
        print(f"Attempting to load from Secrets Manager: {SECRET_NAME}...")
        
        secrets_client = boto3.client('secretsmanager', region_name=os.getenv('AWS_REGION'))
        response = secrets_client.get_secret_value(SecretId=SECRET_NAME)
        secrets = json.loads(response['SecretString'])
        
        os.environ['ELASTIC_CLOUD_ID'] = secrets['ELASTIC_CLOUD_ID']
        os.environ['ELASTIC_USERNAME'] = secrets.get('ELASTIC_USERNAME', 'elastic')
        os.environ['ELASTIC_PASSWORD'] = secrets['ELASTIC_PASSWORD']
        os.environ['ELASTIC_ENDPOINT'] = secrets['ELASTIC_ENDPOINT']
        
        print("✅ Loaded from Secrets Manager!")
        print(f"   Endpoint: {os.getenv('ELASTIC_ENDPOINT')[:50]}...")
        
    except Exception as e:
        print(f"ℹ️  Secrets Manager not configured: {str(e)[:80]}")
        print("   Falling back to direct configuration...\n")
        USE_SECRETS_MANAGER = False

# Option 2: Direct configuration (for workshop)
if not USE_SECRETS_MANAGER:
    print("Using direct configuration (edit the values below):")
    print()
    
    # ⚠️ EDIT THESE VALUES ⚠️
    os.environ['ELASTIC_CLOUD_ID'] = 'your-cloud-id-here'
    os.environ['ELASTIC_USERNAME'] = 'elastic'
    os.environ['ELASTIC_PASSWORD'] = 'your-password-here'
    os.environ['ELASTIC_ENDPOINT'] = 'https://your-deployment.es.us-east-1.aws.found.io:9243'
    
    print("✅ Direct configuration set")
    print("   ⚠️  Remember to update the values above!")

print("\n" + "="*60)
print("\n🎯 Configuration complete! Continue to next cell →")
print("\n💡 Tip: To use Secrets Manager, run:")
print("   aws secretsmanager create-secret --name elastic-workshop-credentials \\")
print("     --secret-string '{\"ELASTIC_CLOUD_ID\":\"...\", ...}'")

## Step 3: Test Elasticsearch Connection

Let's verify we can connect to your Elastic Cloud deployment from SageMaker:

In [ ]:
from elasticsearch import Elasticsearch
import sys

print("🔌 Testing Elastic Cloud connection...\n")

try:
    # Connect to Elastic Cloud
    es = Elasticsearch(
        cloud_id=os.getenv('ELASTIC_CLOUD_ID'),
        basic_auth=(os.getenv('ELASTIC_USERNAME'), os.getenv('ELASTIC_PASSWORD'))
    )
    
    # Get cluster info
    info = es.info()
    
    print("✅ Successfully connected to Elastic Cloud!\n")
    print("="*60)
    print(f"Cluster Name: {info['cluster_name']}")
    print(f"Version: {info['version']['number']}")
    print(f"Cluster UUID: {info['cluster_uuid']}")
    print(f"Tagline: {info['tagline']}")
    print("="*60)
    
    # Test search capability
    print("\n🔍 Testing search capability...")
    health = es.cluster.health()
    print(f"   Status: {health['status']}")
    print(f"   Nodes: {health['number_of_nodes']}")
    print(f"   Data nodes: {health['number_of_data_nodes']}")
    
    print("\n✅ Elastic Cloud is ready!")
    
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Check credentials in Step 2")
    print("2. Verify deployment is running in Elastic Cloud Console")
    print("3. Check Security → Traffic filters allow SageMaker IP")
    print("\n💡 Get your SageMaker IP:")
    
    try:
        import requests
        my_ip = requests.get('https://api.ipify.org', timeout=5).text
        print(f"   Your IP: {my_ip}")
        print(f"   Add this to Elastic Cloud traffic filter: {my_ip}/32")
    except:
        print("   Could not detect IP")
    
    sys.exit(1)

## Step 4: Verify ELSER Model

Check if ELSER v2 is deployed and running:

In [ ]:
print("🤖 Checking ELSER model status...\n")

try:
    # Check if ELSER model exists
    response = es.ml.get_trained_models(model_id=".elser_model_2")
    
    if response['trained_model_configs']:
        print("✅ ELSER model found!")
        
        model_info = response['trained_model_configs'][0]
        print(f"   Model ID: {model_info['model_id']}")
        print(f"   Model type: {model_info.get('model_type', 'N/A')}")
        
        # Check deployment status
        print("\n🚀 Checking deployment status...")
        stats = es.ml.get_trained_models_stats(model_id=".elser_model_2")
        
        if stats['trained_model_stats']:
            model_stats = stats['trained_model_stats'][0]
            deployment = model_stats.get('deployment_stats', {})
            state = deployment.get('state', 'not_deployed')
            
            print(f"   State: {state}")
            
            if state == 'started':
                print("\n✅ ELSER is deployed and running!")
                print("\n📊 Deployment details:")
                print(f"   Allocations: {deployment.get('number_of_allocations', 0)}")
                print(f"   Threads per allocation: {deployment.get('threads_per_allocation', 0)}")
                print(f"   Queue capacity: {deployment.get('queue_capacity', 0)}")
                
                # Test inference
                print("\n🧪 Testing ELSER inference...")
                test_result = es.ml.infer_trained_model(
                    model_id=".elser_model_2",
                    docs=[{"text_field": "travel to romantic destination"}]
                )
                
                if test_result:
                    tokens = len(test_result['inference_results'][0]['predicted_value'])
                    print(f"   Generated {tokens} sparse vector tokens")
                    print("\n✅ ELSER inference working!")
                
            else:
                print(f"\n⚠️  ELSER state: {state}")
                print("\n🔧 To fix:")
                print("1. Open Kibana → Machine Learning → Trained Models")
                print("2. Find .elser_model_2")
                print("3. Click 'Start deployment'")
                print("4. Wait 1-2 minutes and re-run this cell")
        else:
            print("\n⚠️  ELSER model exists but not deployed")
            print("\n🔧 Deploy in Kibana → ML → Trained Models → .elser_model_2 → Deploy")
    else:
        print("❌ ELSER model not found")
        print("\n🔧 To fix:")
        print("1. Open Kibana → Machine Learning → Trained Models")
        print("2. Find '.elser_model_2' in the list")
        print("3. Click 'Download model' (takes 1-2 minutes)")
        print("4. Click 'Deploy model'")
        print("5. Re-run this cell")
        
except Exception as e:
    print(f"❌ Error checking ELSER: {e}")
    print("\n🔧 Make sure:")
    print("1. Your Elastic deployment has Machine Learning enabled")
    print("2. Deployment has ML nodes (4GB+ RAM recommended)")
    print("3. ELSER model is downloaded and deployed")

## Step 5: Test AWS Bedrock Access

**SageMaker Advantage**: Automatic AWS credentials via IAM role!

In [ ]:
import boto3

print("🔐 Testing AWS Bedrock access...\n")
print("(Using IAM role from SageMaker - no access keys needed!)\n")

try:
    # Create Bedrock client (credentials from IAM role)
    bedrock = boto3.client('bedrock', region_name=os.getenv('AWS_REGION'))
    
    # List available foundation models
    print("📋 Listing available models...")
    models = bedrock.list_foundation_models()
    
    # Filter for Claude models
    claude_models = [
        m for m in models['modelSummaries'] 
        if 'claude' in m['modelId'].lower()
    ]
    
    print(f"\n✅ AWS Bedrock access verified!")
    print(f"\n📊 Available Claude models: {len(claude_models)}\n")
    print("="*60)
    
    for model in claude_models[:5]:  # Show first 5
        print(f"• {model['modelId']}")
        if 'responseStreamingSupported' in model:
            print(f"  Streaming: {model['responseStreamingSupported']}")
    
    print("="*60)
    
    if len(claude_models) == 0:
        print("\n⚠️  No Claude models accessible")
        print("\n🔧 To fix:")
        print("1. Go to AWS Console → Amazon Bedrock → Model access")
        print("2. Click 'Request model access'")
        print("3. Select Anthropic Claude models")
        print("4. Submit (usually instant approval)")
        print("5. Wait 2-5 minutes and re-run this cell")
    else:
        print("\n✅ Ready to use Claude via Bedrock!")
        
except Exception as e:
    print(f"❌ Error accessing Bedrock: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Check SageMaker IAM role has Bedrock permissions")
    print("2. Verify you're in correct region (us-east-1)")
    print("3. Confirm model access is granted in Bedrock console")
    print("\n💡 Required IAM permission: bedrock:ListFoundationModels")

## Step 6: Test Claude Invocation

Let's do a quick test of Claude via Bedrock:

In [ ]:
import json

print("🤖 Testing Claude via AWS Bedrock...\n")

try:
    bedrock_runtime = boto3.client(
        'bedrock-runtime',
        region_name=os.getenv('AWS_REGION')
    )
    
    # Simple test prompt
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 150,
        "messages": [
            {
                "role": "user",
                "content": "Say 'Hello from Claude on SageMaker!' and confirm you're ready to help with travel planning using Elastic ELSER. Keep it brief (2 sentences)."
            }
        ],
        "temperature": 0.7
    }
    
    print("📤 Sending request to Claude 3.5 Sonnet...")
    
    response = bedrock_runtime.invoke_model(
        modelId="anthropic.claude-3-5-sonnet-20240620-v1:0",
        body=json.dumps(request_body)
    )
    
    response_body = json.loads(response['body'].read())
    message = response_body['content'][0]['text']
    
    print("\n✅ Claude is responding!\n")
    print("="*60)
    print(f"Claude says:\n")
    print(message)
    print("="*60)
    
    # Show usage stats
    usage = response_body.get('usage', {})
    print(f"\n📊 Token usage:")
    print(f"   Input tokens: {usage.get('input_tokens', 'N/A')}")
    print(f"   Output tokens: {usage.get('output_tokens', 'N/A')}")
    
    print("\n✅ Bedrock + Claude working perfectly!")
    
except Exception as e:
    print(f"❌ Error calling Claude: {e}")
    print("\n🔧 Check:")
    print("1. Model access granted for Claude 3.5 Sonnet")
    print("2. IAM role has bedrock:InvokeModel permission")
    print("3. Using correct model ID")

## ✅ Verification Complete!

### What You've Accomplished:

✅ Installed all workshop dependencies in SageMaker  
✅ Configured credentials (IAM role + Elastic)  
✅ Connected to Elastic Cloud from SageMaker  
✅ Verified ELSER model is deployed and running  
✅ Tested AWS Bedrock access via IAM role  
✅ Invoked Claude successfully  

### 🎯 You're Ready for the Workshop!

---

## Next Steps:

1. **Continue to**: `01-ELSER-Semantic-Search.ipynb`
2. **Or review** any ❌ errors above and fix them first

---

## 💡 SageMaker-Specific Tips

### Save Your Work
- SageMaker auto-saves notebooks
- But manually save before stopping: File → Save

### Stop Instance When Not Using
```bash
# Via Console: SageMaker → Notebook instances → Stop
# Or CLI:
aws sagemaker stop-notebook-instance \
  --notebook-instance-name travel-agent-workshop
```

### Monitor Costs
- ml.t3.medium: ~$0.05/hour
- Full workshop: ~$0.20 total
- Bedrock usage: ~$3-5 for full workshop

### Get Help
- SageMaker docs: https://docs.aws.amazon.com/sagemaker/
- Workshop docs: See `../SAGEMAKER_SETUP.md`

---

## 🚀 Ready?

**Open `01-ELSER-Semantic-Search.ipynb` and let's build!**

---

*Running on: Amazon SageMaker*  
*Workshop: Travel Intelligence Agent*  
*Version: 3.1 - SageMaker Edition*